# 03. Data Preprocessing Teks Berita

Notebook ini khusus untuk tahap **Text Preprocessing** terhadap dataset berita `dataset_detik.csv`.
Tahapannya meliputi:
1. **Cleaning & Case Folding**: Mengubah huruf kecil dan menghapus simbol/angka.
2. **Tokenisasi**: Memecah kalimat menjadi daftar kata (token).
3. **Stopword Removal**: Menghilangkan kata umum yang tidak membedakan topik.
4. Menyimpan hasil ke `dataset_preprocessing.csv` untuk tahap TF-IDF.

## 1. Import Library

In [1]:
import re
from collections import Counter
import pandas as pd
from IPython.display import display

## 2. Membaca Dataset
Membaca dataset hasil crawling dari `dataset_detik.csv`.

In [2]:
df = pd.read_csv("dataset_detik.csv")
print("Jumlah data:", len(df))
display(df.head(3))

Jumlah data: 200


,id,isi_berita,label,url
0,SP001,Profil Desak Made Peraih Emas Keempat RI: Kena...,sport,https://sport.detik.com/sport-lain/d-8686722/p...
1,SP002,"Gacor di Asian Games, Leo/Daniel Kini Bidik Tu...",sport,https://sport.detik.com/raket/d-8686185/gacor-...
2,SP003,Asian Games 2026: Desak Made Raih Emas Keempat...,sport,https://sport.detik.com/sport-lain/d-8686648/a...


## 3. Preprocessing Teks (Cleaning, Tokenisasi & Stopword Removal)
Menghapus angka dan karakter non-alfabet, memecah kata (minimal 3 huruf), dan menyaring stopwords.

In [3]:
# Daftar stopword esensial bahasa Indonesia
STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan",
    "pada", "dalam", "atau", "juga", "akan", "ada", "sebagai", "lebih",
    "sudah", "telah", "tidak", "karena", "oleh", "saat", "hingga", "dapat",
    "bisa", "jadi", "para", "nya", "ia", "tak", "tersebut", "mereka",
    "tautan", "disalin", "artikel", "baca", "saya", "selengkapnya", "detik"
}

def preprocess_text(teks):
    # 1. Hapus simbol & angka (hanya huruf dan spasi)
    teks_bersih = re.sub(r"[^a-zA-Z\s]", " ", str(teks).lower())
    # 2. Tokenisasi kata minimal 3 huruf
    tokens = re.findall(r"[a-zA-Z]{3,}", teks_bersih)
    # 3. Stopword removal
    tokens_bersih = [kata for kata in tokens if kata not in STOPWORDS]
    # 4. Gabungkan kembali ke teks bersih
    return " ".join(tokens_bersih)

# Terapkan preprocessing ke seluruh berita
df["berita_clean"] = df["isi_berita"].apply(preprocess_text)
df["jumlah_kata_asli"] = df["isi_berita"].apply(lambda x: len(str(x).split()))
df["jumlah_kata_clean"] = df["berita_clean"].apply(lambda x: len(str(x).split()))

print("Total kata sebelum preprocessing:", df["jumlah_kata_asli"].sum())
print("Total kata setelah preprocessing:", df["jumlah_kata_clean"].sum())
display(df[["id", "label", "jumlah_kata_asli", "jumlah_kata_clean"]].head())

Total kata sebelum preprocessing: 75328
Total kata setelah preprocessing: 56860


,id,label,jumlah_kata_asli,jumlah_kata_clean
0,SP001,sport,436,330
1,SP002,sport,361,286
2,SP003,sport,241,181
3,SP004,sport,433,340
4,SP005,sport,459,325


## 4. Pemeriksaan Hasil Preprocessing
Melihat perbandingan sebelum dan sesudah serta 10 kata yang paling sering muncul.

In [4]:
print("=== CONTOH PERBANDINGAN DATA 1 ===")
print("Sebelum:")
print(df.loc[0, "isi_berita"][:250], "...")
print("\nSesudah:")
print(df.loc[0, "berita_clean"][:250], "...")

# 10 Kata Terbanyak
semua_kata = " ".join(df["berita_clean"]).split()
top_10 = Counter(semua_kata).most_common(10)
df_top10 = pd.DataFrame(top_10, columns=["Kata", "Frekuensi"])
print("\n=== 10 KATA PALING SERING MUNCUL ===")
display(df_top10)

=== CONTOH PERBANDINGAN DATA 1 ===
Sebelum:
Profil Desak Made Peraih Emas Keempat RI: Kenal Panjat Tebing karena Bibi Mercy Raya - detikSport Rabu, 30 Sep 2026 20:15 WIB Jakarta - Desak Made Rita Kusuma Dewi kembali bikin bangga Indonesia. Ia menjadi atlet tercepat se-Asia setelah mempertahank ...

Sesudah:
profil desak made peraih emas keempat kenal panjat tebing bibi mercy raya detiksport rabu sep wib jakarta desak made rita kusuma dewi kembali bikin bangga indonesia menjadi atlet tercepat asia setelah mempertahankan medali emasnya asian games aichi n ...

=== 10 KATA PALING SERING MUNCUL ===


,Kata,Frekuensi
0,indonesia,633
1,games,520
2,asian,512
3,emas,381
4,menjadi,289
5,medali,288
6,jakarta,280
7,wib,273
8,leo,251
9,daniel,246


## 5. Simpan Dataset Hasil Preprocessing
Menyimpan hasil ke file `dataset_preprocessing.csv` untuk digunakan pada tahap berikutnya (TF-IDF).

In [5]:
df.to_csv("dataset_preprocessing.csv", index=False)
print("Data hasil preprocessing berhasil disimpan ke 'dataset_preprocessing.csv'.")
display(df[["id", "label", "berita_clean"]].head())

Data hasil preprocessing berhasil disimpan ke 'dataset_preprocessing.csv'.


,id,label,berita_clean
0,SP001,sport,profil desak made peraih emas keempat kenal pa...
1,SP002,sport,gacor asian games leo daniel kini bidik tur er...
2,SP003,sport,asian games desak made raih emas keempat indon...
3,SP004,sport,panjat tebing buru medali yenni wahid aditya v...
4,SP005,sport,mau nonton fia rallycross world cup indonesia ...
